# Исследование надежности заемщиков


Во второй части проекта вы выполните шаги 3 и 4. Их вручную проверит ревьюер.
Чтобы вам не пришлось писать код заново для шагов 1 и 2, мы добавили авторские решения в ячейки с кодом. 



## Откройте таблицу и изучите общую информацию о данных

**Задание 1. Импортируйте библиотеку pandas. Считайте данные из csv-файла в датафрейм и сохраните в переменную `data`. Путь к файлу:**

`/datasets/data.csv`

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

try:
    data = pd.read_csv('/datasets/data.csv')
except:
    data = pd.read_csv('https://code.s3.yandex.net/datasets/data.csv')

**Задание 2. Выведите первые 20 строчек датафрейма `data` на экран.**

In [3]:
data.head(20)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
5,0,-926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419,покупка жилья
6,0,-2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920,операции с жильем
7,0,-152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197,образование
8,2,-6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424,на проведение свадьбы
9,0,-2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277,покупка жилья для семьи


**Задание 3. Выведите основную информацию о датафрейме с помощью метода `info()`.**

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


## Предобработка данных

### Удаление пропусков

**Задание 4. Выведите количество пропущенных значений для каждого столбца. Используйте комбинацию двух методов.**

In [5]:
data.isna().sum()

children               0
days_employed       2174
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2174
purpose                0
dtype: int64

**Задание 5. В двух столбцах есть пропущенные значения. Один из них — `days_employed`. Пропуски в этом столбце вы обработаете на следующем этапе. Другой столбец с пропущенными значениями — `total_income` — хранит данные о доходах. На сумму дохода сильнее всего влияет тип занятости, поэтому заполнить пропуски в этом столбце нужно медианным значением по каждому типу из столбца `income_type`. Например, у человека с типом занятости `сотрудник` пропуск в столбце `total_income` должен быть заполнен медианным доходом среди всех записей с тем же типом.**

In [6]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['total_income'].isna()), 'total_income'] = \
    data.loc[(data['income_type'] == t), 'total_income'].median()

### Обработка аномальных значений

**Задание 6. В данных могут встречаться артефакты (аномалии) — значения, которые не отражают действительность и появились по какой-то ошибке. таким артефактом будет отрицательное количество дней трудового стажа в столбце `days_employed`. Для реальных данных это нормально. Обработайте значения в этом столбце: замените все отрицательные значения положительными с помощью метода `abs()`.**

In [7]:
data['days_employed'] = data['days_employed'].abs()

**Задание 7. Для каждого типа занятости выведите медианное значение трудового стажа `days_employed` в днях.**

In [8]:
data.groupby('income_type')['days_employed'].agg('median')

income_type
безработный        366413.652744
в декрете            3296.759962
госслужащий          2689.368353
компаньон            1547.382223
пенсионер          365213.306266
предприниматель       520.848083
сотрудник            1574.202821
студент               578.751554
Name: days_employed, dtype: float64

У двух типов (безработные и пенсионеры) получатся аномально большие значения. Исправить такие значения сложно, поэтому оставьте их как есть. Тем более этот столбец не понадобится вам для исследования.

**Задание 8. Выведите перечень уникальных значений столбца `children`.**

In [9]:
data['children'].unique()

array([ 1,  0,  3,  2, -1,  4, 20,  5])

**Задание 9. В столбце `children` есть два аномальных значения. Удалите строки, в которых встречаются такие аномальные значения из датафрейма `data`.**

In [10]:
data = data[(data['children'] != -1) & (data['children'] != 20)]

**Задание 10. Ещё раз выведите перечень уникальных значений столбца `children`, чтобы убедиться, что артефакты удалены.**

In [11]:
data['children'].unique()

array([1, 0, 3, 2, 4, 5])

### Удаление пропусков (продолжение)

**Задание 11. Заполните пропуски в столбце `days_employed` медианными значениями по каждого типа занятости `income_type`.**

In [12]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['days_employed'].isna()), 'days_employed'] = \
    data.loc[(data['income_type'] == t), 'days_employed'].median()

**Задание 12. Убедитесь, что все пропуски заполнены. Проверьте себя и ещё раз выведите количество пропущенных значений для каждого столбца с помощью двух методов.**

In [13]:
data.isna().sum()

children            0
days_employed       0
dob_years           0
education           0
education_id        0
family_status       0
family_status_id    0
gender              0
income_type         0
debt                0
total_income        0
purpose             0
dtype: int64

### Изменение типов данных

**Задание 13. Замените вещественный тип данных в столбце `total_income` на целочисленный с помощью метода `astype()`.**

In [14]:
data['total_income'] = data['total_income'].astype(int)

### Обработка дубликатов

**Задание 14. Обработайте неявные дубликаты в столбце `education`. В этом столбце есть одни и те же значения, но записанные по-разному: с использованием заглавных и строчных букв. Приведите их к нижнему регистру. Проверьте остальные столбцы.**

In [15]:
data['education'] = data['education'].str.lower()

**Задание 15. Выведите на экран количество строк-дубликатов в данных. Если такие строки присутствуют, удалите их.**

In [16]:
data.duplicated().sum()

71

In [17]:
data = data.drop_duplicates()

### Категоризация данных

**Задание 16. На основании диапазонов, указанных ниже, создайте в датафрейме `data` столбец `total_income_category` с категориями:**

- 0–30000 — `'E'`;
- 30001–50000 — `'D'`;
- 50001–200000 — `'C'`;
- 200001–1000000 — `'B'`;
- 1000001 и выше — `'A'`.


**Например, кредитополучателю с доходом 25000 нужно назначить категорию `'E'`, а клиенту, получающему 235000, — `'B'`. Используйте собственную функцию с именем `categorize_income()` и метод `apply()`.**

In [18]:
def categorize_income(income):
    try:
        if 0 <= income <= 30000:
            return 'E'
        elif 30001 <= income <= 50000:
            return 'D'
        elif 50001 <= income <= 200000:
            return 'C'
        elif 200001 <= income <= 1000000:
            return 'B'
        elif income >= 1000001:
            return 'A'
    except:
        pass

In [19]:
data['total_income_category'] = data['total_income'].apply(categorize_income)

**Задание 17. Выведите на экран перечень уникальных целей взятия кредита из столбца `purpose`.**

In [20]:
data['purpose'].unique()

array(['покупка жилья', 'приобретение автомобиля',
       'дополнительное образование', 'сыграть свадьбу',
       'операции с жильем', 'образование', 'на проведение свадьбы',
       'покупка жилья для семьи', 'покупка недвижимости',
       'покупка коммерческой недвижимости', 'покупка жилой недвижимости',
       'строительство собственной недвижимости', 'недвижимость',
       'строительство недвижимости', 'на покупку подержанного автомобиля',
       'на покупку своего автомобиля',
       'операции с коммерческой недвижимостью',
       'строительство жилой недвижимости', 'жилье',
       'операции со своей недвижимостью', 'автомобили',
       'заняться образованием', 'сделка с подержанным автомобилем',
       'получение образования', 'автомобиль', 'свадьба',
       'получение дополнительного образования', 'покупка своего жилья',
       'операции с недвижимостью', 'получение высшего образования',
       'свой автомобиль', 'сделка с автомобилем',
       'профильное образование', 'высшее об

**Задание 18. Создайте функцию, которая на основании данных из столбца `purpose` сформирует новый столбец `purpose_category`, в который войдут следующие категории:**

- `'операции с автомобилем'`,
- `'операции с недвижимостью'`,
- `'проведение свадьбы'`,
- `'получение образования'`.

**Например, если в столбце `purpose` находится подстрока `'на покупку автомобиля'`, то в столбце `purpose_category` должна появиться строка `'операции с автомобилем'`.**

**Используйте собственную функцию с именем `categorize_purpose()` и метод `apply()`. Изучите данные в столбце `purpose` и определите, какие подстроки помогут вам правильно определить категорию.**

In [21]:
def categorize_purpose(row):
    try:
        if 'автом' in row:
            return 'операции с автомобилем'
        elif 'жил' in row or 'недвиж' in row:
            return 'операции с недвижимостью'
        elif 'свад' in row:
            return 'проведение свадьбы'
        elif 'образов' in row:
            return 'получение образования'
    except:
        return 'нет категории'

In [22]:
data['purpose_category'] = data['purpose'].apply(categorize_purpose)

### Шаг 3. Исследуйте данные и ответьте на вопросы

#### 3.1 Есть ли зависимость между количеством детей и возвратом кредита в срок?

In [42]:
# Ваш код будет здесь. Вы можете создавать новые ячейки.

# Создание сводной таблицы с подсчетами
ch = data.pivot_table(index='children', values='debt', aggfunc=['count', 'sum', 'mean'])

# Переименуем колонки для удобства
ch.columns = ['Количество заемщиков', 'Количество должников', 'Процент просрочивших кредит']

# Сортируем таблицу по 'Процент просрочивших кредит' и форматируем его как проценты
ch = ch.sort_values(by='Количество заемщиков', ascending=False)
ch['Процент просрочивших кредит'] = ch['Процент просрочивших кредит'].apply(lambda x: "{:.2%}".format(x))

ch

,Количество заемщиков,Количество должников,Процент просрочивших кредит
children,,,
0,14091,1063,7.54%
1,4808,444,9.23%
2,2052,194,9.45%
3,330,27,8.18%
4,41,4,9.76%
5,9,0,0.00%


**Вывод:** 

* 8.12% клиентов допускают просрочки по выплате кредитов.
* Клиенты без детей чаще других выплачивают кредит в срок (7.54% должники).
* Клиенты, у которых 3+ детей реже допускают просрочку, чем немногодетные семьи.
* Клиенты с 1 или 2 детьми чаще всего не выплачивали кредит в срок (9.23% и 9.45% должники соответственно).

#### 3.2 Есть ли зависимость между семейным положением и возвратом кредита в срок?

In [40]:
# Ваш код будет здесь. Вы можете создавать новые ячейки.
data['family_status'].unique()
# Смотрим какие виды семейного положения имеются в наших данных

array(['женат / замужем', 'гражданский брак', 'вдовец / вдова',
       'в разводе', 'Не женат / не замужем'], dtype=object)

In [49]:
def percent(x):
    return "{0:.2%}".format(x) # сделаем функцию для удобства

In [50]:
fs = data.pivot_table(index = ['family_status'], values='debt', aggfunc = ('mean', lambda X:
                                                                      X.count()))

fs['mean'] = fs['mean'].apply(percent)
fs = fs.rename(columns={"<lambda_0>": "Количество записей", "mean": "Среднее"})
fs.sort_values(by='Количество записей', ascending=False)

,Количество записей,Среднее
family_status,,
женат / замужем,12261,7.56%
гражданский брак,4134,9.31%
Не женат / не замужем,2796,9.76%
в разводе,1189,7.06%
вдовец / вдова,951,6.62%


**Вывод:** 
Тут уже у нас вырисовывается некий тренд жизненного пути. Больше всего должников мы наблюдаем в категории "не женатых", чуть меньше клиентов, которые относятся к категории "гражданский брак", еще меньше случаем долгов в категории "женатых" и еще меньше у тех кто "в разводе" и "вдовец".

**Таким образом, не женатые клиенты имеют повышенный риск не возвратить кредит вовремя, а клиенты, кторые находятся или находились в браке более надежные** 

#### 3.3 Есть ли зависимость между уровнем дохода и возвратом кредита в срок?

In [51]:
# Ваш код будет здесь. Вы можете создавать новые ячейки.
data_debt = data[data['debt'] == 1] # имел долги
data_not_debt = data[data['debt'] == 0] # не имел долги
data_debt['total_income'].describe() # глянем статистику по доходу должника

count    1.732000e+03
mean     1.611523e+05
std      9.792373e+04
min      2.066700e+04
25%      1.081488e+05
50%      1.425940e+05
75%      1.875995e+05
max      2.200852e+06
Name: total_income, dtype: float64

In [52]:
data_not_debt['total_income'].describe() # глянем статистику по доходу не должника

count    1.959900e+04
mean     1.657131e+05
std      9.834230e+04
min      2.120500e+04
25%      1.074405e+05
50%      1.425940e+05
75%      1.966145e+05
max      2.265604e+06
Name: total_income, dtype: float64

Ранее мы сделали разделение на категории по уровня дохода клиентов:
* 0–30 000 — 'E';
* 30 001–50 000 — 'D';
* 50 001–200 000 — 'C';
* 200 001–1 000 000 — 'B';
* 1 000 001 и выше — 'A'.

In [53]:
profit_client = data.pivot_table(index = ['total_income_category'], values='debt', aggfunc = ('mean', lambda X:
                                                                      X.count()))

profit_client['mean'] = profit_client['mean'].apply(percent)
profit_client = profit_client.rename(columns={"<lambda_0>": "Количество записей", "mean": "Средняя задолженость"})
profit_client.sort_values(by='Количество записей', ascending=False)

,Количество записей,Средняя задолженость
total_income_category,,
C,15921,8.50%
B,5014,7.06%
D,349,6.02%
A,25,8.00%
E,22,9.09%


**Вывод:**

* Самые богатые (доход более 1 млн) и самые бедные клиенты (доход не более 30 тыс.) относительно часто не возвращают кредит в срок (8.00% и 9.09% соответственно)

* Категория С не оплачивает кредит вовремя в 8.50% случаев

* Клиенты с не самым высоким доходом. Например клиенты категории В не оплачивают кредит вовремя в 7.06% случаев

* Категория D, невовремя оплачивает кредиты в 6.02% случаев

**Таким образом, самые надежные клиенты имеют доход немного выше среднего (200 001–1 000 000) или средний (50 001–200 000)** 


#### 3.4 Как разные цели кредита влияют на его возврат в срок?

In [56]:
# Ваш код будет здесь. Вы можете создавать новые ячейки.
data['purpose_category'].unique()

array(['операции с недвижимостью', 'операции с автомобилем',
       'получение образования', 'проведение свадьбы'], dtype=object)

In [57]:
p_с = data.pivot_table(index = ['purpose_category'], values='debt', aggfunc = ('mean', lambda X:
                                                                      X.count()))

p_с['mean'] = p_с['mean'].apply(percent)
p_с = p_с.rename(columns={"<lambda_0>": "Количество записей", "mean": "Средняя задолженость"})
p_с.sort_values(by='Количество записей', ascending=False)

,Количество записей,Средняя задолженость
purpose_category,,
операции с недвижимостью,10751,7.26%
операции с автомобилем,4279,9.35%
получение образования,3988,9.25%
проведение свадьбы,2313,7.91%


**Вывод:** 

Не удивительно, но меньше всего должников у приобретателей недвижимости, чуть больше должников среди тех, кто проводил свадьбу. Сильно больше должников среди тех, кто брал кредит получение образования и самая большая доля должников находится в категории тех, кто покупал автомобиль в кредит

#### 3.5 Приведите возможные причины появления пропусков в исходных данных.

*Ответ:* Возможно часть клиентов с пропусками (это около 10%) работают не официально как это часто бывает и не могли подтвердить свой стаж работы и доход по основному месту работы либо это техническая ошибка и часть данных просто не перенеслась из какой-то системы

<div class="alert alert-success"; style="border-left: 7px solid green">
<h4> ✔️ <font color="green">Комментарий ревьюера (зеленый)</font></h4>

Да, основные причины — технические ошибки и человеческий фактор.
    
</div> 

#### 3.6 Объясните, почему заполнить пропуски медианным значением — лучшее решение для количественных переменных.

*Ответ:* 
Медианное значение меньше исказит действительность, чем средние значения т.к. среднее арифметическое при своем расчете включает и самые большие значения и самы маленькие и если этот разбро сильно велик, а каличество таких экстремальных случае невелико6 то использование среднего арифметического сильно поменяет картину происходящего потому что большинство остальных данных будет сильно отличаться от этого среднего значения. А если использовать медианное значение, которое равняется такому значению, которое есть в 50% строк наших данных, то и искажение этих значений снизится потому что его подсчете не будут включаться супербольшие данные, которые при подсчете сред. арифметического нам искажали данные

<div class="alert alert-success"; style="border-left: 7px solid green">
<h5> ✔️ <font color="green">Комментарий ревьюера (зеленый)</font></h5>

Верно, медиана, в отличие от среднего значения, менее чувствительна к выбросам, среднее значение смещается в сторону выбросов.

</div>

### Шаг 4: общий вывод.

Напишите ваш общий вывод.

В процессе предобработки данных мы наблюдали отсутствие некоторых значений в части дохода и стажа работы у некоторых клиентов. Недостающие данные мы заполнили медианным значением, чтобы при анализе получить более точны значения. Полагаем, что указанные данные отсутствовали либо из-за технической ошибки либо из-за человечкого фактора (клиенты не имели официального трудоустройства и не могли документально подтвердить свой доход и стаж). 

После заполнения отсутствующих значений, для ранжирования клиентов мы разделили их на группы в зависимости от доходов, семейного положения и количества детей, а также цели кредитования.
По резульатам данной предобработки мы оценили как влияют вышеуказанные критерии на возврат займов и заметили следующее.

8.12% клиентов допускают просрочки по выплате кредитов.

После проведенного анализа мы можем дать портрет надежному клиенту в плане своевременного возрата кредита, а также описать некоторые признаки не надежного клиента:

* children - портретом идеального клиента является семья без детей. Худшим клиентом является семья у которой 1-2 ребенка.


* family_status - портретом идеального клиента является женатый и даже бывший в браке клиент. Худшим клиентом является не женатый.


* total_income - портретом идеального клиента является клиент c доходом немного выше среднего (200 001–1 000 000) или средним (50 001–200 000). Худшим клиентом является: самые бедные (доход не более 30 тыс.) или самые богатые клиенты (доход более 1 млн).


* purpose - портретом идеального клиента является цель кредитования для покупки недвижимости. Худшей целью для банка является кредитование для приобретения автомобиля.

<div class="alert alert-success"; style="border-left: 7px solid green">
<h4> ✔️ <font color="green"> v3 Комментарий ревьюера (зеленый)</font></h4>

Спасибо,хороший вывод, который содержит все промежуточные результаты 🔥.
    
</div> 